# Model test & validation: Linear Regression

Floor baseline. Expected to perform poorly overall, and to be numerically well-behaved (no conditioning issues) since it's the simplest possible model on only 3 standardized features.

In [ ]:
from sklearn.linear_model import LinearRegression

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))

df_raw = pd.read_csv("../../data/chf_long_clean.csv")
df = df_raw[df_raw.X != 1.0].reset_index(drop=True)
FEATURES = ["P", "G", "X"]
TARGET = "CHF"
sorted_P = sorted(df.P.unique())

# Split A (random, seed 0) -- quick interpolation check
X_all, y_all = df[FEATURES].values, df[TARGET].values
XtrA, XteA, ytrA, yteA = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

# Split C (edge extrapolation) -- the honest test
train_dfC = df[df.P <= 16000].reset_index(drop=True)
test_dfC = df[df.P >= 17000].reset_index(drop=True)
XtrC, ytrC = train_dfC[FEATURES].values, train_dfC[TARGET].values
XteC, yteC = test_dfC[FEATURES].values, test_dfC[TARGET].values

print(f"Split A: {len(XtrA)} train / {len(XteA)} test")
print(f"Split C: {len(XtrC)} train / {len(XteC)} test")


## Fit on Split A (interpolation) and Split C (extrapolation)

In [ ]:

scaler = StandardScaler().fit(XtrA)
model_A = LinearRegression().fit(scaler.transform(XtrA), np.log(ytrA))
predA = np.exp(model_A.predict(scaler.transform(XteA)))
print(f"Split A: R2={r2(yteA, predA):.4f}, MAPE={mape(yteA, predA):.2f}%")

scalerC = StandardScaler().fit(XtrC)
model_C = LinearRegression().fit(scalerC.transform(XtrC), np.log(ytrC))
predC = np.exp(model_C.predict(scalerC.transform(XteC)))
print(f"Split C: R2={r2(yteC, predC):.4f}, MAPE={mape(yteC, predC):.2f}%")


## Edge-case tests

Check the design matrix's condition number (multicollinearity risk -- low expected, since P/G/X are grid-sampled and largely independent) and the learned coefficients' relative magnitudes (which input dominates the linear fit).

In [ ]:

X_design = scaler.transform(XtrA)
cond_number = np.linalg.cond(np.column_stack([X_design, np.ones(len(X_design))]))
print(f"Design matrix condition number: {cond_number:.2f} (low = well-conditioned; "
      f"large [>1e6] would indicate multicollinearity risk)")
print(f"Coefficients (log-CHF per std-dev of P, G, X): {model_A.coef_}")
print(f"Intercept: {model_A.intercept_:.4f}")
assert cond_number < 100, "Unexpectedly ill-conditioned design matrix for only 3 near-independent grid features"
print("PASS: design matrix well-conditioned, as expected for 3 grid-sampled inputs.")


## Diagnostic plot

In [ ]:

residuals = np.log(yteC) - model_C.predict(scalerC.transform(XteC))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(yteA, predA, s=5, alpha=0.3)
lims = [0, max(yteA.max(), predA.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_title(f"Split A parity, R2={r2(yteA,predA):.3f}")
axes[0].set_xlabel("True CHF"); axes[0].set_ylabel("Predicted CHF")
axes[1].hist(residuals, bins=40)
axes[1].set_title("Split C residuals (log-CHF scale)")
axes[1].set_xlabel("log(true) - predicted log(CHF)")
plt.tight_layout()
plt.savefig("../results/model_tests_linear.png", dpi=100)
plt.show()
